In [2]:
import numpy as np
import matplotlib.pyplot as plt
import sys,os
import math
from pathlib import Path
parent_dir = Path.cwd().parent
sys.path.append(str(parent_dir))
#sys.path.append(r"\Users\13347\Documents\Yale\Paraproducts\Quasilinearization-of-potential-and-Schrodinger-kernels-with-tensor-paraproducts-main")
from tensor_paraproducts import haar_paraproduct
import pickle
import time

# TPA matrix-vector multiplication XPs 

In [3]:

def fcn_gen(dim):
    # Gaussian
    mu = 0.5
    sigma = 0.015
    xone = np.linspace(0, 1, dim)
    vone = np.exp(-0.5 * ((xone - mu) / sigma) ** 2)
    
    # Sine
    xtwo = np.linspace(0, (2 * np.pi)*10, dim)
    vtwo = np.sin(xtwo)
    
    # Power fcn
    vthree=[]
    for i in xone:
        vthree.append(dim * (i / dim) ** 3)
    vthree = vthree/np.max(np.abs(vthree))

    return [vone,vtwo,vthree]
    

In [7]:
XPdict = {}
dimlist = [128,256,512]
deltlist = [0.003,0.03,0.3]
for N in dimlist:

    # Generate coordinates
    hill_x = np.linspace(0.4, 0.6, N)
    hill_y = np.exp(-((hill_x - 0.5) / .1) ** 2) *.6
    
    xloc = 0.61
    line_x = np.full(N, xloc)
    line_y = np.linspace(0, 0.4, N)
    
    hill_x = np.expand_dims(hill_x,axis=1)
    hill_y = np.expand_dims(hill_y,axis=1)
    line_x = np.expand_dims(line_x,axis=1)
    line_y = np.expand_dims(line_y,axis=1)
    hill = np.concatenate((hill_x,hill_y),axis=1)
    line = np.concatenate((line_x,line_y),axis=1)
    
    # Compute distance matrix
    dxy = np.zeros((N,N))
    for i in range(N):
        for j in range(N):
            dxy[i,j] = np.linalg.norm(hill[i] - line[j])
    
    # Compute polynomial
    deg = 5
    poly = dxy**(-(deg+1))

    # Compute potential kernel
    Apot = np.log(poly + 1e-16)
    
    # Normalization
    npoly = poly/np.max(np.abs(poly))
    nApot = Apot/np.max(np.abs(Apot))

    # TPA construction
    jlist = [int(math.log2(N)) - 1,int(math.log2(N)),int(math.log2(N)) + 1]
    hp = haar_paraproduct(npoly,nApot)
    hp.kernel = 'conpot'

    for jmax in jlist:
        tpaof = hp.fast_tpa(jmax)
        resid = nApot - tpaof
        eps = 2**(-jmax)

        for delt in deltlist:

            exlist = fcn_gen(N)
            
            for k in range(len(exlist)):
    
                ntpaof = tpaof/np.max(np.abs(tpaof))
                mask = np.abs(ntpaof) > delt
                tpaof_delt = tpaof * mask

                tfcn = exlist[k]
                gt = nApot @ exlist[k]
                approx_delt = (tpaof_delt + resid) @ tfcn
                
                l2 = np.linalg.norm(gt - approx_delt)/np.linalg.norm(gt)
                linf = np.max(np.abs(gt - approx_delt))/np.max(np.abs(gt))
                cr = (N*N)/(np.sum(mask))
                
                metrics = [l2,linf,cr]
                rawdata = [gt,approx_delt,tfcn]
                params = [eps,delt,N]
                rdata = [metrics,rawdata,params]
                strg = str(N) + "_" + str(eps) + "_" + str(delt) + "fcn_" + str(k)
                XPdict[strg] = rdata
                #print("Test")

                
    


In [8]:
#print(XPdict[strg])

with open("XPdict.pkl", "wb") as f:
    pickle.dump(XPdict, f)